# 高温作业专用服装设计 — 复现

本 Notebook 按题目三个问题分块复现，底层求解器使用 C-N 差分。
如果计算较慢，可先减小时间长度或增大步长做试跑。

In [8]:
from pathlib import Path
import numpy as np

from model import (
    load_appendix_xlsx,
    solve_heat_cn,
    steady_state_h_in,
    rmse,
    write_simple_xlsx,
)

ROOT = Path('.').resolve()
APPENDIX = ROOT / 'CUMCM-2018-Problem-A-Chinese-Appendix.xlsx'

In [9]:
data = load_appendix_xlsx(APPENDIX)
data.materials, data.thickness_range_mm, data.measured_time_s[:5], data.measured_temp_c[-5:]

({'I': {'rho': 300.0, 'c': 1377.0, 'k': 0.082},
  'II': {'rho': 862.0, 'c': 2100.0, 'k': 0.37},
  'III': {'rho': 74.2, 'c': 1726.0, 'k': 0.045},
  'IV': {'rho': 1.18, 'c': 1005.0, 'k': 0.028}},
 {'I': (0.6, 0.6), 'II': (0.6, 25.0), 'III': (3.6, 3.6), 'IV': (0.6, 6.4)},
 array([0., 1., 2., 3., 4.]),
 array([48.08, 48.08, 48.08, 48.08, 48.08]))

## 问题一：参数反演与拟合

环境温度 75°C、II 层 6 mm、IV 层 5 mm、工作 90 分钟。
目标：反演 $h_I$ 与 $h_{IV}$，并输出温度场 `problem1.xlsx`。

In [10]:
materials = data.materials
thickness = {
    'I': 0.6,
    'II': 6.0,
    'III': 3.6,
    'IV': 5.0,
}

t_env = 75.0
t_skin = 37.0
t_surface = float(data.measured_temp_c[-1])

def estimate_h_in(h_out: float) -> float:
    return steady_state_h_in(
        materials=materials,
        thickness_mm=thickness,
        t_env=t_env,
        t_skin=t_skin,
        t_surface=t_surface,
        h_out=h_out,
    )

def fit_h_out(h_out_values: np.ndarray) -> dict:
    best = {'h_out': None, 'h_in': None, 'rmse': float('inf')}
    for h_out in h_out_values:
        h_in = estimate_h_in(h_out)
        _, temps = solve_heat_cn(
            materials=materials,
            thickness_mm=thickness,
            t_env=t_env,
            h_out=h_out,
            h_in=h_in,
            t_skin=t_skin,
            total_time_s=int(data.measured_time_s[-1]),
            dx_mm=0.1,
            dt_s=1.0,
            store_all=False,
        )
        sim_skin = temps[:, 0]
        score = rmse(sim_skin, data.measured_temp_c)
        if score < best['rmse']:
            best = {'h_out': h_out, 'h_in': h_in, 'rmse': score}
    return best

h_out_grid = np.arange(110.0, 120.01, 0.2)
best = fit_h_out(h_out_grid)
best

{'h_out': np.float64(110.0),
 'h_in': np.float64(8.343541806262863),
 'rmse': 0.08851205788803088}

In [11]:
# 使用最佳参数重算全温度场并导出 problem1.xlsx
h_out = float(best['h_out'])
h_in = float(best['h_in'])
times, temps = solve_heat_cn(
    materials=materials,
    thickness_mm=thickness,
    t_env=t_env,
    h_out=h_out,
    h_in=h_in,
    t_skin=t_skin,
    total_time_s=int(data.measured_time_s[-1]),
    dx_mm=0.1,
    dt_s=1.0,
    store_all=True,
)

# 构造空间坐标（mm）
dx_mm = 0.1
counts = [
    int(round(thickness['I'] / dx_mm)),
    int(round(thickness['II'] / dx_mm)),
    int(round(thickness['III'] / dx_mm)),
    int(round(thickness['IV'] / dx_mm)),
]
n = sum(counts)
x_mm = np.arange(n) * dx_mm

header = ['time_s'] + [f'x_{x:.2f}_mm' for x in x_mm]
out = np.column_stack([times, temps])
write_simple_xlsx(ROOT / 'problem1.xlsx', header, out)
ROOT / 'problem1.xlsx'

PosixPath('/Users/Zhuanz1/Desktop/code/数模/problem1.xlsx')

## 问题二：单变量定步长搜索

环境温度 65°C、IV 层 5.5 mm、工作 60 分钟。
约束：皮肤温度 ≤47°C 且 >44°C 时间 ≤300 s。

In [17]:
t_env = 65.0
thickness_p2 = {
    'I': 0.6,
    'II': 6.0,
    'III': 3.6,
    'IV': 5.5,
}
total_time_s = 3600

def check_constraints(d2_mm: float) -> dict:
    thickness_p2['II'] = d2_mm
    h_out = float(best['h_out'])
    h_in = steady_state_h_in(
        materials=materials,
        thickness_mm=thickness_p2,
        t_env=t_env,
        t_skin=t_skin,
        t_surface=t_surface,
        h_out=h_out,
    )
    _, temps = solve_heat_cn(
        materials=materials,
        thickness_mm=thickness_p2,
        t_env=t_env,
        h_out=h_out,
        h_in=h_in,
        t_skin=t_skin,
        total_time_s=total_time_s,
        dx_mm=0.1,
        dt_s=1.0,
        store_all=False,
    )
    skin = temps[:, 0]
    max_temp = float(np.max(skin))
    over_44 = float(np.sum(skin > 44.0))
    over_time = over_44 * 1.0
    ok = (max_temp <= 47.0) and (over_time <= 300.0)
    return {'d2': d2_mm, 'ok': ok, 'max': max_temp, 'over_44_s': over_time}

coarse = np.arange(10.0, 25.01, 1.0)
coarse_results = [check_constraints(d2) for d2 in coarse]
feasible_coarse = [r for r in coarse_results if r['ok']]
print('feasible coarse:', feasible_coarse[:5], 'count:', len(feasible_coarse))
print('min max_temp in coarse:', min(r['max'] for r in coarse_results))

feasible coarse: [] count: 0
min max_temp in coarse: 46.4540525146997


In [18]:
fine = np.arange(15.0, 22.01, 0.1)
fine_results = [check_constraints(d2) for d2 in fine]
feasible = [r for r in fine_results if r['ok']]
if feasible:
    best_p2 = min(feasible, key=lambda r: r['d2'])
    print('best_p2:', best_p2)
else:
    print('no feasible d2 in scan')
    print('min max_temp in fine:', min(r['max'] for r in fine_results))

no feasible d2 in scan
min max_temp in fine: 47.088682893542256


## 问题三：二维区域搜索

环境温度 80°C、工作 30 分钟。
目标：先最小化 $d_2$，再最小化平衡时间。

In [ ]:
t_env = 80.0
total_time_s = 1800

def settle_time(skin: np.ndarray, dt: float = 1.0, tol: float = 0.05, window: int = 300) -> float:
    target = float(skin[-1])
    for i in range(len(skin) - window):
        segment = skin[i:i + window]
        if np.all(np.abs(segment - target) <= tol):
            return i * dt
    return float(len(skin) * dt)

def check_pair(d2_mm: float, d4_mm: float) -> dict:
    thickness_p3 = {
        'I': 0.6,
        'II': d2_mm,
        'III': 3.6,
        'IV': d4_mm,
    }
    h_out = float(best['h_out'])
    h_in = steady_state_h_in(
        materials=materials,
        thickness_mm=thickness_p3,
        t_env=t_env,
        t_skin=t_skin,
        t_surface=t_surface,
        h_out=h_out,
    )
    _, temps = solve_heat_cn(
        materials=materials,
        thickness_mm=thickness_p3,
        t_env=t_env,
        h_out=h_out,
        h_in=h_in,
        t_skin=t_skin,
        total_time_s=total_time_s,
        dx_mm=0.1,
        dt_s=1.0,
        store_all=False,
    )
    skin = temps[:, 0]
    max_temp = float(np.max(skin))
    over_44 = float(np.sum(skin > 44.0))
    over_time = over_44 * 1.0
    ok = (max_temp <= 47.0) and (over_time <= 300.0)
    return {
        'd2': d2_mm,
        'd4': d4_mm,
        'ok': ok,
        'max': max_temp,
        'over_44_s': over_time,
        'settle_s': settle_time(skin),
    }

d2_grid = np.arange(18.0, 21.01, 0.1)
d4_grid = np.arange(0.6, 6.41, 0.1)
feasible = []
for d2 in d2_grid:
    for d4 in d4_grid:
        r = check_pair(d2, d4)
        if r['ok']:
            feasible.append(r)

best_p3 = min(feasible, key=lambda r: (r['d2'], r['settle_s']))
best_p3